# Create Prediction Inputs

Build the target and the readings from 1 and 24 hours earlier.

Open this notebook in `notebooks/M3/`. Run the cells in order. Saving again replaces this notebook's output files.


## Shared folder setup

In [ ]:
from pathlib import Path
PROJECT_ROOT = Path("../..").resolve()
PROJECT_ROOT

## Read the hourly file

In [ ]:
import pandas as pd
hourly = pd.read_csv(PROJECT_ROOT / "data/pilot/hourly_sample.csv")
hourly.head()

## Keep the three columns we need

In [ ]:
hourly = hourly[["LCLid", "hour_start", "kWh"]]
hourly.columns = ["LCLid", "hour", "kwh"]
hourly.head()

## Read the hour as a date and time

In [ ]:
hourly["hour"] = pd.to_datetime(hourly["hour"], utc=True)
hourly.head()

## List the expected hours

In [ ]:
hours = pd.date_range(hourly["hour"].min(), hourly["hour"].max(), freq="h")
hours[:5]

## Make a household-and-hour grid

In [ ]:
homes = sorted(hourly["LCLid"].unique())
grid = pd.MultiIndex.from_product([homes, hours], names=["LCLid", "target_hour"])
len(grid)

## Index the observed values

In [ ]:
values = hourly.set_index(["LCLid", "hour"])["kwh"]
values.index.names = ["LCLid", "target_hour"]
values.head()

## Include absent hours

In [ ]:
values = values.reindex(grid)
values.head()

## Add the target

In [ ]:
all_rows = values.reset_index(name="target_kwh")
all_rows.head()

## Find the previous hour

In [ ]:
previous_time = all_rows["target_hour"] - pd.Timedelta(hours=1)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], previous_time])
previous_values = values.reindex(keys)
all_rows["lag_1h_kwh"] = previous_values.values
all_rows.head()

## Find the same hour yesterday

In [ ]:
yesterday_time = all_rows["target_hour"] - pd.Timedelta(hours=24)
keys = pd.MultiIndex.from_arrays([all_rows["LCLid"], yesterday_time])
yesterday_values = values.reindex(keys)
all_rows["lag_24h_kwh"] = yesterday_values.values
all_rows.iloc[22:27]

## Add calendar information

In [ ]:
all_rows["hour"] = all_rows["target_hour"].dt.hour
all_rows["weekday"] = all_rows["target_hour"].dt.dayofweek
all_rows.iloc[22:27]

## Find rows with missing values

In [ ]:
all_rows["row_status"] = "kept"
all_rows.loc[all_rows["lag_24h_kwh"].isna(), "row_status"] = "missing_lag_24h"
all_rows.loc[all_rows["lag_1h_kwh"].isna(), "row_status"] = "missing_lag_1h"
all_rows.loc[all_rows["target_kwh"].isna(), "row_status"] = "missing_target"
all_rows["row_status"].value_counts()

## Keep the usable rows

In [ ]:
features = all_rows[all_rows["row_status"] == "kept"]
features = features.drop(columns="row_status")
features.head()

## Count the rows

In [ ]:
len(features)

## Count missing values by home

In [ ]:
counts = pd.crosstab(all_rows["LCLid"], all_rows["row_status"])
counts = counts.reindex(columns=["kept", "missing_target",
    "missing_lag_1h", "missing_lag_24h"], fill_value=0)
counts.columns = ["rows_kept", "excluded_missing_target",
    "excluded_missing_lag_1h", "excluded_missing_lag_24h"]
counts["grid_rows"] = all_rows.groupby("LCLid").size()
counts

## Count missing fields separately

In [ ]:
for field, name in [("target_kwh", "any_target_missing"),
                    ("lag_1h_kwh", "any_lag_1h_missing"), ("lag_24h_kwh", "any_lag_24h_missing")]:
    counts[name] = all_rows[field].isna().groupby(all_rows["LCLid"]).sum()
counts

## Count observed targets on the first day

In [ ]:
first_hour = all_rows.groupby("LCLid")["target_hour"].transform("min")
first_day = all_rows["target_hour"] < first_hour + pd.Timedelta(hours=24)
observed = first_day & all_rows["target_kwh"].notna()
counts["target_observed_in_first_24h"] = observed.groupby(all_rows["LCLid"]).sum()
counts

## Finish the count file

In [ ]:
column_order = ["grid_rows", "rows_kept", "excluded_missing_target", "excluded_missing_lag_1h",
                "excluded_missing_lag_24h", "any_target_missing", "any_lag_1h_missing",
                "any_lag_24h_missing", "target_observed_in_first_24h"]
counts = counts[column_order].copy()
counts.loc["TOTAL"] = counts.sum()
counts = counts.reset_index()
counts

## Create a new folder for this run

In [ ]:
from pathlib import Path
output_dir = PROJECT_ROOT / 'data/processed'
output_dir.mkdir(parents=True, exist_ok=True)
output_dir

## Save the features

In [ ]:
feature_path = output_dir / "prediction_inputs_sample.csv"
features.to_csv(feature_path, index=False)
feature_path

## Save the counts

In [ ]:
counts_path = output_dir / "prediction_input_counts_sample.csv"
counts.to_csv(counts_path, index=False)
counts_path